# AI Stock Prediction (KNN) — Fixed Version
Original used Quandl (discontinued free API) with a missing data-fetch step. This version uses `yfinance` instead — free, no API key needed.

In [ ]:
!pip install yfinance -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score

In [ ]:
TICKER = "AAPL"
data = yf.download(TICKER, start="2015-01-01", end="2025-01-01", auto_adjust=True)
data.columns = data.columns.get_level_values(0) if isinstance(data.columns, pd.MultiIndex) else data.columns
data.head(10)

In [ ]:
plt.figure(figsize=(16,8))
plt.plot(data['Close'], label='Closing Price')
plt.title(f'{TICKER} Closing Price')
plt.legend()
plt.show()

In [ ]:
data['Open - Close'] = data['Open'] - data['Close']
data['High - Low'] = data['High'] - data['Low']
data = data.dropna()

X = data[['Open - Close','High - Low']]
X.head()

In [ ]:
Y_class = np.where(data['Close'].shift(-1) > data['Close'], 1, -1)
Y_class

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y_class, test_size=0.25, random_state=44)

In [ ]:
params = {'n_neighbors':[2,3,4,5,6,7,8,9,10,11,12,13,14,15]}
knn = KNeighborsClassifier()
model = GridSearchCV(knn, params, cv=5)
model.fit(X_train, Y_train)

predictions_classification = model.predict(X_test)
accuracy_train = accuracy_score(Y_train, model.predict(X_train))
accuracy_test = accuracy_score(Y_test, predictions_classification)

print('Train_data Accuracy: %.2f' % accuracy_train)
print('Test_data Accuracy: %.2f' % accuracy_test)
print('Best k:', model.best_params_)

In [ ]:
actual_predicted_data = pd.DataFrame({'Actual class': Y_test, 'Predicted Class': predictions_classification})
actual_predicted_data.head(10)

In [ ]:
Y_reg = data['Close']
X_train_reg, X_test_reg, Y_train_reg, Y_test_reg = train_test_split(X, Y_reg, test_size=0.25, random_state=44)

knn_reg = KNeighborsRegressor()
model_reg = GridSearchCV(knn_reg, params, cv=5)
model_reg.fit(X_train_reg, Y_train_reg)
predictions = model_reg.predict(X_test_reg)
print(predictions[:10])

In [ ]:
valid = pd.DataFrame({'Actual Close': Y_test_reg.values, 'Predicted Close': predictions})
valid.head(10)